In [ ]:
!pip install langchain langchain-core langchain-nvidia-ai-endpoints pydantic gradio

In [ ]:
import os
from typing import Optional

from pydantic import BaseModel, Field

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableMap,
    RunnablePassthrough,
)

from langchain_core.runnables.passthrough import RunnableAssign

from langchain_core.output_parsers import PydanticOutputParser

import gradio as gr
from pprint import pprint

In [ ]:
os.environ["NVIDIA_API_KEY"] = ""

In [ ]:
def get_flight_info(d: dict) -> str:
    req_keys = ['first_name', 'last_name', 'confirmation']
    assert all((key in d) for key in req_keys)

    keys = req_keys + ["departure", "destination", "departure_time", "arrival_time", "flight_day"]
    values = [
        ["Karina", "Sarkar", 12345, "Kolkata", "New Zealand", "12:30 PM", "9:30 PM", "tomorrow"],
        ["John", "Wick", 54321, "New York", "Los Angeles", "8:00 AM", "11:00 AM", "Sunday"],
        ["Alice", "Wonder", 98765, "Chicago", "Miami", "7:00 PM", "11:00 PM", "next week"],
        ["Bobbi", "Brown", 56789, "Chicago", "Seattle", "1:00 PM", "4:00 PM", "yesterday"],
    ]

    get_key = lambda d: "|".join([d['first_name'], d['last_name'], str(d['confirmation'])])
    get_val = lambda l: {k:v for k,v in zip(keys, l)}

    db = {get_key(get_val(entry)): get_val(entry) for entry in values}

    data = db.get(get_key(d))
    if not data:
        return "No flight info found. Ask user to confirm details."

    return (
        f"{data['first_name']} {data['last_name']}'s flight from {data['departure']} to {data['destination']}"
        f" departs at {data['departure_time']} {data['flight_day']} and lands at {data['arrival_time']}."
    )

In [ ]:
print(get_flight_info({"first_name" : "Karina", "last_name" : "Sarkar", "confirmation" : 12345}))

Karina Sarkar's flight from Kolkata to New Zealand departs at 12:30 PM tomorrow and lands at 9:30 PM.


In [ ]:
class KnowledgeBase(BaseModel):
    first_name: str = Field('unknown')
    last_name: str = Field('unknown')
    confirmation: Optional[int] = Field(None)
    discussion_summary: str = Field("")
    open_problems: str = Field("")
    current_goals: str = Field("")

In [ ]:
external_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a chatbot for SkyFlow Airlines, and you are helping a customer with their issue."
        " Please chat with them! Stay concise and clear!"
        " Your running knowledge base is: {know_base}."
        " This is for you only; Do not mention it!"
        " \nUsing that, we retrieved the following: {context}\n"
        " If they provide info and the retrieval fails, ask to confirm their first/last name and confirmation."
        " Do not ask them any other personal info."
        " If it's not important to know about their flight, do not ask."
        " The checking happens automatically; you cannot check manually."
    )),
    ("assistant", "{output}"),
    ("user", "{input}"),
])

parser_prompt = ChatPromptTemplate.from_template(
    "Extract structured info.\n{format_instructions}\n\n"
    "OLD: {know_base}\n"
    "ASSISTANT: {output}\n"
    "USER: {input}\n"
    "NEW:"
)

In [ ]:
chat_llm = ChatNVIDIA(model="meta/llama-3.1-70b-instruct") | StrOutputParser()
instruct_llm = ChatNVIDIA(model="mistralai/mixtral-8x22b-instruct-v0.1") | StrOutputParser()

In [ ]:
parser = PydanticOutputParser(pydantic_object=KnowledgeBase)

def extract_knowledge(state):
    prompt = parser_prompt.partial(
        format_instructions=parser.get_format_instructions()
    )

    chain = prompt | instruct_llm

    result = chain.invoke({
        "know_base": state.get("know_base"),
        "input": state.get("input"),
        "output": state.get("output"),
    })

    try:
        return parser.parse(result)
    except:
        return state.get("know_base", KnowledgeBase())

In [ ]:
def retrieve_db(state):
    kb = state["know_base"]
    user_input = state.get("input", "").lower()

    # Only trigger DB if user asks about flight-related things
    trigger_words = ["flight", "ticket", "departure", "arrival", "airport"]

    if not any(word in user_input for word in trigger_words):
        return "No flight query detected."

    # If flight query but missing info
    if (
        kb.first_name == "unknown" or
        kb.last_name == "unknown" or
        kb.confirmation in [None, -1]
    ):
        return "Missing flight info. Ask user for first name, last name, and confirmation number."

    # Otherwise retrieve
    return get_flight_info({
        "first_name": kb.first_name,
        "last_name": kb.last_name,
        "confirmation": kb.confirmation
    })

In [ ]:
internal_chain = (
    RunnableAssign({'know_base': extract_knowledge})
    | RunnableAssign({'context': retrieve_db})
)

In [ ]:
external_chain = external_prompt | chat_llm

In [ ]:
state = {'know_base': KnowledgeBase()}

def chat_gen(message, history=[]):
    global state

    state['input'] = message
    state['history'] = history
    state['output'] = "" if not history else history[-1][1]

    state = internal_chain.invoke(state)

    print("\nSTATE:")
    pprint(state)

    response = external_chain.invoke(state)
    return response

In [ ]:
history = []

while True:
    msg = input("\nYou: ")
    reply = chat_gen(msg, history)
    print("Bot:", reply)
    history.append([msg, reply])


You: Hello, how is it going?

STATE:
{'context': 'No flight query detected.',
 'history': [],
 'input': 'Hello, how is it going?',
 'know_base': KnowledgeBase(first_name='unknown', last_name='unknown', confirmation=None, discussion_summary='', open_problems='', current_goals=''),
 'output': ''}
Bot: Hello, I'm doing well, thank you. How can I assist you with your travel plans or concerns with SkyFlow Airlines today?

You: Can you tell me a bit about skyflow?

STATE:
{'context': 'No flight query detected.',
 'history': [['Hello, how is it going?',
              "Hello, I'm doing well, thank you. How can I assist you with "
              'your travel plans or concerns with SkyFlow Airlines today?']],
 'input': 'Can you tell me a bit about skyflow?',
 'know_base': KnowledgeBase(first_name='unknown', last_name='unknown', confirmation=None, discussion_summary='', open_problems='', current_goals=''),
 'output': "Hello, I'm doing well, thank you. How can I assist you with your "
           '

KeyboardInterrupt: Interrupted by user